# Import libraries and the dataset

In [1]:
import sys
!{sys.executable} -m pip install xgboost

In [2]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

In [3]:
df=pd.read_csv('truedata.csv')

In [4]:
df = df.dropna()

In [5]:
df

,Strain Rate,Temperature,True Strain,True Stress,True Plastic Strain
0,0.0001,27.0,0.10683,939.31764,0.00030
1,0.0001,27.0,0.10683,939.32333,0.00030
2,0.0001,27.0,0.10684,939.33680,0.00031
3,0.0001,27.0,0.10684,939.34085,0.00031
4,0.0001,27.0,0.10685,939.34472,0.00032
...,...,...,...,...,...
162754,0.0100,500.0,0.06196,597.07372,0.01804
162755,0.0100,500.0,0.06218,597.09635,0.01826
162756,0.0100,500.0,0.06239,597.11009,0.01847
162757,0.0100,500.0,0.06260,597.11561,0.01868


In [6]:
TargetVariable=['True Stress']
Predictors=['Strain Rate','Temperature','True Plastic Strain']
 
X=df[Predictors].values
y=df[TargetVariable].values

# Hyperparameter tuning using GridSearchCV where k=10 on the entire dataset

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test  = train_test_split(X, y, test_size=0.3, random_state=42)

In [10]:
params = { 'max_depth': [3,6,9,10,12,15],
           'learning_rate': [0.01, 0.05, 0.1, 0.3, 0.5],
           'n_estimators': [10, 50, 100, 200, 500],
           'colsample_bytree': [0.3, 0.7]}
xgbr = xgb.XGBRegressor(seed = 20)
clf = GridSearchCV(estimator=xgbr, 
                   param_grid=params,
                   scoring='neg_mean_squared_error', 
                   verbose=1,cv=10,n_jobs=-1)
clf.fit(X_train, y_train)
print("Best parameters:", clf.best_params_)

Fitting 10 folds for each of 300 candidates, totalling 3000 fits
Best parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.3, 'max_depth': 6, 'n_estimators': 500}


In [11]:
clf.best_estimator_

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.3, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             n_estimators=500, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=None, ...)

In [12]:
clf.best_score_

-31.057658821930186

# Refitting and retraining the model with the best parameters and evaluating its performance

In [8]:
xgbr = xgb.XGBRegressor(colsample_bytree=0.7, learning_rate=0.3, max_depth=6,
             n_estimators=500, seed=20)
xgbr.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.3, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             n_estimators=500, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=None, ...)

In [25]:
y_pred = xgbr.predict(X_test)

In [26]:
y_pred

array([ 779.7011 ,  622.9113 ,  692.4424 , ...,  749.18964, 1152.1765 ,
        950.4944 ], dtype=float32)

In [27]:
len(y_pred)

48828

In [28]:
y_test = y_test.ravel()

In [29]:
y_test

array([ 781.83656,  622.26907,  694.36165, ...,  740.82561, 1152.24326,
        950.02404])

In [30]:
APE=100*(abs(y_test-y_pred)/y_test)

In [31]:
import numpy as np

In [15]:
print('The Accuracy of XGB model is:', 100-np.mean(APE))

The Accuracy of XGB model is: 99.63252750360631


In [35]:
y_pred = y_pred.tolist()

In [36]:
y_pred

[779.7011108398438,
 622.9113159179688,
 692.4423828125,
 793.9641723632812,
 732.2110595703125,
 656.806640625,
 822.4813232421875,
 1174.3475341796875,
 891.5999145507812,
 713.163818359375,
 795.9757080078125,
 962.3287353515625,
 983.7581176757812,
 942.7730712890625,
 684.9339599609375,
 988.6712646484375,
 678.7251586914062,
 793.7557373046875,
 793.0885620117188,
 781.7420654296875,
 743.5505981445312,
 677.4048461914062,
 712.9873657226562,
 1047.257568359375,
 712.19482421875,
 706.74951171875,
 874.21728515625,
 1145.9068603515625,
 944.05810546875,
 751.3970947265625,
 791.7717895507812,
 716.994384765625,
 948.6309814453125,
 784.6118774414062,
 691.506103515625,
 651.917236328125,
 720.3541870117188,
 722.5491333007812,
 694.7766723632812,
 742.6138305664062,
 677.680419921875,
 992.8339233398438,
 943.316162109375,
 618.728515625,
 655.7814331054688,
 881.2210693359375,
 825.5110473632812,
 722.202880859375,
 674.471923828125,
 787.9595336914062,
 790.771240234375,
 697.9

In [37]:
MAE=(abs(y_test-y_pred))
print('The Accuracy of XGB model is:',np.mean(MAE))

The Accuracy of XGB model is: 2.8287815738324293


In [38]:
import statistics
var = (statistics.variance(y_pred))
print(var)
chi_sq = np.sum(((y_test-y_pred)**2)/var)
red_chi_sq = chi_sq/5074
print('The reduced chi squared value for RFR is', red_chi_sq)

23014.346177207437
The reduced chi squared value for RFR is 0.01247646753470998


# Validating the model

In [19]:
dfvalid=pd.read_csv('150Cvalidationdata.csv')

In [20]:
Predictors=['Strain Rate','Temperature','True Plastic Strain']
X_valid=dfvalid[Predictors].values

In [21]:
X_valid

array([[1.0000000e-03, 1.5000000e+02, 6.1896600e-05],
       [1.0000000e-03, 1.5000000e+02, 5.2687700e-05],
       [1.0000000e-03, 1.5000000e+02, 6.9798000e-05],
       ...,
       [1.0000000e-03, 1.5000000e+02, 9.4723457e-02],
       [1.0000000e-03, 1.5000000e+02, 9.4695481e-02],
       [1.0000000e-03, 1.5000000e+02, 9.4743924e-02]])

In [22]:
y_pred_valid = xgbr.predict(X_valid)

In [23]:
y_pred_valid

array([624.8972 , 624.8972 , 624.8972 , ..., 756.3892 , 756.3892 ,
       757.15405], dtype=float32)

In [24]:
import pandas as pd 
pd.DataFrame(y_pred_valid).to_csv("xgbvalidationpredicteddata.csv")